## Step 1: Stationarity Check — Full (3-Type Detection)

### Types Handled
| # | Type | Fix Applied |
|---|------|-------------|
| 1 | **Stochastic trend** (unit root) | Regular differencing `d` |
| 2 | **Deterministic trend** | Detrend first, then difference |
| 3 | **Seasonal non-stationarity** | Seasonal differencing `D` |

### Tests Used
| Test | Null Hypothesis | Rejects when |
|------|----------------|---------------|
| **ADF** | Unit root exists | `p < 0.05` → stationary |
| **KPSS** `regression="c"` | Series is stationary (level) | `p < 0.05` → non-stationary |
| **KPSS** `regression="ct"` | Series is stationary around trend | `p < 0.05` → deterministic trend |
| **ACF at lag s** | — | `ACF > 0.4` → seasonal unit root suspected |

### Decision Logic
```
Stationary   ←  ADF rejects (p < 0.05)  AND  KPSS level fails to reject (p > 0.05)
Unit root    ←  ADF fails to reject      OR   KPSS level rejects
Det. trend   ←  KPSS with trend term still significant (p < 0.05)
Seasonal UR  ←  ACF at lag s > 0.4  (strong seasonal memory)
```

### Order of Operations in `make_stationary_full()`
```
1. Detrend         →  remove deterministic drift first
2. Seasonal diff   →  remove seasonal unit root  Y(t) - Y(t-s)
3. Regular diff    →  remove remaining stochastic trend
```
> **Why this order?** Doing regular differencing before seasonal differencing can obscure the seasonal structure, making the seasonal unit root harder to detect and remove cleanly.

### Outputs Tracked Per Column
| Variable | Meaning | Used in |
|----------|---------|--------|
| `diff_order[col]` | `d` — regular differences applied | Step 6 `d_range` |
| `seas_order[col]` | `D` — seasonal differences applied | Step 6 `D_range` |
| `trend_removed[col]` | Whether deterministic trend was removed | Summary |
| `D_RANGE` | Auto-set from `D_max` for grid search | Step 6 |

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from statsmodels.tsa.stattools     import adfuller, kpss, acf
from statsmodels.graphics.tsaplots import plot_acf

### Configuration

In [ ]:
# ── Season length ────────────────────────────────────────────────
# Monthly data  → SEASON_PERIOD = 12
# Quarterly     → SEASON_PERIOD = 4
# Weekly        → SEASON_PERIOD = 52
SEASON_PERIOD    = 12

# ACF at seasonal lag above this → seasonal unit root suspected
SEASONAL_ACF_THR = 0.4

# Maximum regular differences to attempt before giving up
MAX_DIFFS        = 2

# Minimum observations to run any test
MIN_OBS          = 10

### Diagnostic Plots
Run this **before any transformation** to visually identify which type(s) of non-stationarity are present.

| Row | Plot | What to look for |
|-----|------|------------------|
| 0 | Raw series + rolling mean | Upward/downward drift = deterministic trend |
| 1 | ACF up to 2 seasons | Spike at lag `s` = seasonal unit root |
| 2 | Rolling variance | Non-flat = variance non-stationarity |

In [ ]:
def plot_stationarity_diagnostics(df, features, target, period=SEASON_PERIOD):
    """
    3-row diagnostic grid for every feature and the target.
    Row 0 : Raw series + rolling mean  (deterministic trend check)
    Row 1 : ACF up to 2 seasons        (seasonal unit root check)
    Row 2 : Rolling variance           (variance stability check)
    """
    cols      = features + [target]
    n         = len(cols)
    fig, axes = plt.subplots(3, n, figsize=(4 * n, 10))
    fig.suptitle(
        "Step 1 — Stationarity Diagnostics (raw data)",
        fontsize=13, y=1.01
    )

    for i, col in enumerate(cols):
        s   = df[col].dropna()
        win = max(3, len(s) // 6)

        # ── Row 0: Raw series + rolling mean ─────────────────────
        ax = axes[0, i] if n > 1 else axes[0]
        ax.plot(s.values, linewidth=0.9, color="#378ADD", label="series")
        ax.plot(
            s.rolling(win).mean().values,
            color="#E24B4A", linewidth=1.2, linestyle="--", label="rolling mean"
        )
        ax.set_title(f"{col}\nRaw series", fontsize=9)
        ax.legend(fontsize=7)
        if i == 0:
            ax.set_ylabel("Value", fontsize=9)

        # ── Row 1: ACF up to 2 seasons ────────────────────────────
        ax       = axes[1, i] if n > 1 else axes[1]
        max_lags = min(2 * period, len(s) // 2 - 1)
        plot_acf(
            s, lags=max_lags, ax=ax,
            color="#1D9E75", vlines_kwargs={"colors": "#1D9E75"},
            alpha=0.05, zero=False
        )
        ax.axvline(
            period, color="#E24B4A", linewidth=1.0,
            linestyle="--", alpha=0.8, label=f"lag {period}"
        )
        ax.set_title(f"ACF  (check lag {period})", fontsize=9)
        ax.legend(fontsize=7)
        if i == 0:
            ax.set_ylabel("ACF", fontsize=9)

        # ── Row 2: Rolling variance ───────────────────────────────
        ax       = axes[2, i] if n > 1 else axes[2]
        roll_var = s.rolling(win).var()
        ax.plot(roll_var.values, linewidth=0.9, color="#7F77DD")
        ax.axhline(
            roll_var.mean(), color="#E24B4A",
            linewidth=0.8, linestyle="--", alpha=0.7
        )
        ax.fill_between(
            range(len(roll_var)), roll_var.values,
            roll_var.mean(), alpha=0.12, color="#7F77DD"
        )
        ax.set_title("Rolling variance\n(flat = stable)", fontsize=9)
        ax.set_xlabel("Time step", fontsize=8)
        if i == 0:
            ax.set_ylabel("Variance", fontsize=9)

    plt.tight_layout()
    fname = "step1_stationarity_diagnostics.png"
    plt.savefig(fname, dpi=130, bbox_inches="tight")
    plt.show()
    print(f"  [saved] {fname}\n")

### Core Test Functions

In [ ]:
def _is_stationary(s, name=""):
    """
    Combines ADF + KPSS for a robust stationarity verdict.
    Both must agree:
      ADF  rejects unit root     (p < 0.05)
      KPSS fails to reject stationarity (p > 0.05)
    Using both avoids false positives from each test alone.
    """
    try:
        s = np.array(s, dtype=float)
        s = s[~np.isnan(s)]
        if len(s) < MIN_OBS:
            return False
        adf_p  = adfuller(s)[1]
        kpss_p = kpss(s, regression="c", nlags="auto")[1]
        return (adf_p < 0.05) and (kpss_p > 0.05)
    except Exception:
        return False


def _has_deterministic_trend(s, name=""):
    """
    KPSS with regression='ct' tests stationarity around a deterministic
    trend. If it still rejects (p < 0.05), a significant linear trend
    is present and must be explicitly removed before differencing.
    """
    try:
        s = np.array(s, dtype=float)
        s = s[~np.isnan(s)]
        if len(s) < MIN_OBS:
            return False
        kpss_p = kpss(s, regression="ct", nlags="auto")[1]
        return kpss_p < 0.05
    except Exception:
        return False


def _has_seasonal_unit_root(s, period=SEASON_PERIOD, name=""):
    """
    Checks ACF at the seasonal lag. A high value (> SEASONAL_ACF_THR)
    indicates strong seasonal memory — seasonal unit root is suspected.

    Note: A proper test would use HEGY or OCSB, but those require
    large N. ACF at lag s is a practical proxy for short series.

    Returns:
        (has_seasonal_ur: bool, seasonal_acf_value: float)
    """
    try:
        s = np.array(s, dtype=float)
        s = s[~np.isnan(s)]
        if len(s) < 2 * period + MIN_OBS:
            return False, 0.0
        acf_vals     = acf(s, nlags=period + 1, fft=True)
        seasonal_acf = abs(acf_vals[period])
        return seasonal_acf > SEASONAL_ACF_THR, round(float(seasonal_acf), 3)
    except Exception:
        return False, 0.0

### `make_stationary_full()` — Main Transformation Function

In [ ]:
def make_stationary_full(series, col_name, period=SEASON_PERIOD, max_diffs=MAX_DIFFS):
    """
    Makes a series stationary by detecting and handling all three
    non-stationarity types in the correct order:
      1. Deterministic trend  → detrend first
      2. Seasonal unit root   → seasonal difference  Y(t) - Y(t-s)
      3. Stochastic trend     → regular difference   Y(t) - Y(t-1)

    Returns:
        s           — transformed series (pd.Series)
        d           — regular differences applied
        D           — seasonal differences applied (0 or 1)
        detrended   — True if deterministic trend was removed
    """
    s         = series.copy()
    d         = 0
    D         = 0
    detrended = False
    s_clean   = s.dropna()

    if len(s_clean) < MIN_OBS:
        print(f"  ⚠  {col_name}: too few observations ({len(s_clean)}) — skipping")
        return s, 0, 0, False

    # ── Already stationary? ──────────────────────────────────────
    if _is_stationary(s_clean, col_name):
        print(f"  ✅ {col_name}: already stationary  (d=0, D=0)")
        return s, 0, 0, False

    # ── Step 1: Remove deterministic trend ──────────────────────
    if _has_deterministic_trend(s_clean, col_name):
        t         = np.arange(len(s))
        mask      = s.notna()
        coef      = np.polyfit(t[mask], s[mask], 1)
        s         = s - pd.Series(np.polyval(coef, t), index=s.index)
        detrended = True
        print(f"  ✂  {col_name}: deterministic trend removed")

    # ── Step 2: Seasonal difference ──────────────────────────────
    has_seas, seas_acf = _has_seasonal_unit_root(s.dropna(), period, col_name)
    if has_seas:
        s = s.diff(period)        # Y(t) - Y(t-s)
        D = 1
        print(f"  📅 {col_name}: seasonal difference applied  "
              f"(D=1, s={period}, ACF@lag{period}={seas_acf})")
    else:
        if len(s.dropna()) >= 2 * period + MIN_OBS:
            print(f"  ✅ {col_name}: no seasonal unit root  "
                  f"(ACF@lag{period}={seas_acf})")

    # ── Step 3: Regular differencing ────────────────────────────
    for i in range(max_diffs + 1):
        if _is_stationary(s.dropna(), col_name):
            d = i
            break
        if i < max_diffs:
            s = s.diff()
    else:
        print(f"  ⚠  {col_name}: not stationary after {max_diffs} diffs — kept at d=1")
        s = series.diff()
        d = 1

    tag = f"d={d}, D={D}" + (", detrended" if detrended else "")
    print(f"  ✅ {col_name}: stationary  ({tag})")

    return s, d, D, detrended

### Run Step 1

In [ ]:
# ── Visual diagnostics first — inspect before any transformation ─
print("Generating stationarity diagnostic plots ...")
plot_stationarity_diagnostics(df, ALL_FEATURES, TARGET, period=SEASON_PERIOD)

In [ ]:
print("=" * 60)
print("STEP 1: STATIONARITY CHECK")
print("=" * 60)

# ── Transform all features ───────────────────────────────────────
print("\nTransforming features:\n")

df_stationary  = df.copy()
non_stationary = []
diff_order     = {}    # d per feature
seas_order     = {}    # D per feature
trend_removed  = {}    # detrended flag per feature

for col in ALL_FEATURES:
    transformed, d, D, det = make_stationary_full(df[col], col)
    diff_order[col]        = d
    seas_order[col]        = D
    trend_removed[col]     = det

    if d > 0 or D > 0 or det:
        df_stationary[col] = transformed
        non_stationary.append(col)

        is_now_stat = _is_stationary(transformed.dropna(), col)
        if not is_now_stat:
            print(f"  ⚠  {col}: still not stationary after transformation")

# ── Transform target separately (used in Steps 2–5 only) ────────
print(f"\nTransforming target: {TARGET}")
target_stat, target_d, target_D, target_det = make_stationary_full(
    df[TARGET].copy(), TARGET
)

# ── Align lengths after differencing ────────────────────────────
df_stationary = df_stationary.dropna().reset_index(drop=True)
target_stat   = target_stat.dropna().reset_index(drop=True)
min_len       = min(len(df_stationary), len(target_stat))
df_stationary = df_stationary.iloc[:min_len].reset_index(drop=True)
target_stat   = target_stat.iloc[:min_len].reset_index(drop=True)

# ── d_max / D_max → inform SARIMAX grid search ranges ───────────
d_max = max(diff_order.values()) if diff_order else 0
D_max = max(seas_order.values()) if seas_order else 0

# Set D_RANGE here — used directly in Step 6 grid search
D_RANGE = list(range(D_max + 1))    # [0] or [0, 1]

# ── Summary ──────────────────────────────────────────────────────
d0        = sum(1 for v in diff_order.values() if v == 0)
d1        = sum(1 for v in diff_order.values() if v == 1)
d2        = sum(1 for v in diff_order.values() if v == 2)
n_seas    = sum(1 for v in seas_order.values()   if v > 0)
n_detrend = sum(1 for v in trend_removed.values() if v)

print(f"\n{'─'*60}")
print(f"STEP 1 SUMMARY")
print(f"{'─'*60}")
print(f"Already stationary      (d=0) : {d0}")
print(f"Stationary after 1 diff (d=1) : {d1}")
print(f"Stationary after 2 diff (d=2) : {d2}")
print(f"Seasonal differenced    (D=1) : {n_seas}")
print(f"Deterministic detrended       : {n_detrend}")
print(f"Rows after transformation     : {len(df_stationary)}")
print(f"d_max  (→ informs SARIMAX d)  : {d_max}")
print(f"D_max  (→ informs SARIMAX D)  : {D_max}")
print(f"D_RANGE for grid search       : {D_RANGE}")
print(f"\nPer-feature transformation summary:")
print(f"  {'Feature':<40} {'d':>3}  {'D':>3}  {'detrended':>10}")
print(f"  {'─'*60}")
for col in ALL_FEATURES:
    print(f"  {col:<40} {diff_order[col]:>3}  {seas_order[col]:>3}  "
          f"{'yes' if trend_removed[col] else 'no':>10}")
print(f"\nTarget ({TARGET}):")
print(f"  d={target_d}, D={target_D}, detrended={target_det}")